# 2 - ANALYSE DE DONNEE

Etant donnée que les jeux de données sont tous séparé en "train/test/rul" 

In [37]:
import numpy as np
import pandas as pd
import sklearn as skl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, mean_absolute_error

In [38]:
# Usine N°1
DF_U1_Train = pd.read_csv("Data/csv/train_usine1.csv")
DF_U1_Test = pd.read_csv("Data/csv/test_usine1.csv")
DF_U1_Rul = pd.read_csv("Data/csv/rul_usine1.csv")

# Usine N°2
DF_U2_Train = pd.read_csv("Data/csv/train_usine2.csv")
DF_U2_Test = pd.read_csv("Data/csv/test_usine2.csv")
DF_U2_Rul = pd.read_csv("Data/csv/rul_usine2.csv")

DF_U1_Train

,machine_id,cycle,Altitude / Mach,Throttle Resolver Angle (TRA),Altitude pressurisée,Température entrée fan — T2,Température sortie LPC — T24,Température sortie HPC — T30,Température sortie LPT — T50,Pression entrée fan — P2,...,Bypass Ratio — BPR,Richesse carburant — farB,Soutirage HPC — htBleed,Consigne vitesse fan — Nf_dmd,Consigne vitesse corrigée — PCNfR,Débit refroid. HPT — W31,Débit refroid. LPT — W32,usine,ligne_production,machine_uid
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8.4195,0.03,392,2388,100.0,39.06,23.4190,1,1,1_1_1
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8.4318,0.03,392,2388,100.0,39.00,23.4236,1,1,1_1_1
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8.4178,0.03,390,2388,100.0,38.95,23.3442,1,1,1_1_1
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8.3682,0.03,392,2388,100.0,38.88,23.3739,1,1,1_1_1
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8.4294,0.03,393,2388,100.0,38.90,23.4044,1,1,1_1_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45346,100,148,-0.0016,-0.0003,100.0,518.67,643.78,1596.01,1424.11,14.62,...,8.5036,0.03,394,2388,100.0,38.44,22.9631,1,2,100_1_2
45347,100,149,0.0034,-0.0003,100.0,518.67,643.29,1596.38,1429.14,14.62,...,8.5174,0.03,395,2388,100.0,38.50,22.9746,1,2,100_1_2
45348,100,150,-0.0016,0.0004,100.0,518.67,643.84,1604.53,1431.41,14.62,...,8.5223,0.03,396,2388,100.0,38.39,23.0682,1,2,100_1_2
45349,100,151,-0.0023,0.0004,100.0,518.67,643.94,1597.56,1426.57,14.62,...,8.5148,0.03,395,2388,100.0,38.31,23.0753,1,2,100_1_2


## Création du RUL TEST
Maintenant que les fichiers sont chargé il faut faire le RUL pour le train, afin de savoir ce qu'on cherche a prévoir.

In [39]:
# Logique illustrative — adapte les noms de colonnes aux tiens
DF_U1_Train["RUL"] = (
    DF_U1_Train.groupby("machine_uid")["cycle"].transform("max")
    - DF_U1_Train["cycle"]
)

# Plafonnement à 125 (au-delà, la dégradation n'est pas visible dans les capteurs)
DF_U1_Train["RUL"] = DF_U1_Train["RUL"].clip(upper=125)
DF_U1_Train

,machine_id,cycle,Altitude / Mach,Throttle Resolver Angle (TRA),Altitude pressurisée,Température entrée fan — T2,Température sortie LPC — T24,Température sortie HPC — T30,Température sortie LPT — T50,Pression entrée fan — P2,...,Richesse carburant — farB,Soutirage HPC — htBleed,Consigne vitesse fan — Nf_dmd,Consigne vitesse corrigée — PCNfR,Débit refroid. HPT — W31,Débit refroid. LPT — W32,usine,ligne_production,machine_uid,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,0.03,392,2388,100.0,39.06,23.4190,1,1,1_1_1,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,0.03,392,2388,100.0,39.00,23.4236,1,1,1_1_1,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,0.03,390,2388,100.0,38.95,23.3442,1,1,1_1_1,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,0.03,392,2388,100.0,38.88,23.3739,1,1,1_1_1,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,0.03,393,2388,100.0,38.90,23.4044,1,1,1_1_1,125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45346,100,148,-0.0016,-0.0003,100.0,518.67,643.78,1596.01,1424.11,14.62,...,0.03,394,2388,100.0,38.44,22.9631,1,2,100_1_2,4
45347,100,149,0.0034,-0.0003,100.0,518.67,643.29,1596.38,1429.14,14.62,...,0.03,395,2388,100.0,38.50,22.9746,1,2,100_1_2,3
45348,100,150,-0.0016,0.0004,100.0,518.67,643.84,1604.53,1431.41,14.62,...,0.03,396,2388,100.0,38.39,23.0682,1,2,100_1_2,2
45349,100,151,-0.0023,0.0004,100.0,518.67,643.94,1597.56,1426.57,14.62,...,0.03,395,2388,100.0,38.31,23.0753,1,2,100_1_2,1


In [40]:
DF_U1_Train[DF_U1_Train['machine_uid'] == "100_1_2"]

,machine_id,cycle,Altitude / Mach,Throttle Resolver Angle (TRA),Altitude pressurisée,Température entrée fan — T2,Température sortie LPC — T24,Température sortie HPC — T30,Température sortie LPT — T50,Pression entrée fan — P2,...,Richesse carburant — farB,Soutirage HPC — htBleed,Consigne vitesse fan — Nf_dmd,Consigne vitesse corrigée — PCNfR,Débit refroid. HPT — W31,Débit refroid. LPT — W32,usine,ligne_production,machine_uid,RUL
45199,100,1,0.0027,0.0001,100.0,518.67,642.35,1588.64,1403.10,14.62,...,0.03,393,2388,100.0,39.01,23.2238,1,2,100_1_2,125
45200,100,2,-0.0032,0.0004,100.0,518.67,642.82,1594.15,1407.19,14.62,...,0.03,392,2388,100.0,39.09,23.3526,1,2,100_1_2,125
45201,100,3,-0.0010,-0.0002,100.0,518.67,642.89,1581.68,1410.43,14.62,...,0.03,393,2388,100.0,39.10,23.2936,1,2,100_1_2,125
45202,100,4,0.0018,-0.0001,100.0,518.67,641.95,1587.70,1405.07,14.62,...,0.03,392,2388,100.0,39.04,23.2900,1,2,100_1_2,125
45203,100,5,-0.0017,0.0001,100.0,518.67,642.27,1594.07,1407.75,14.62,...,0.03,393,2388,100.0,38.75,23.3449,1,2,100_1_2,125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45346,100,148,-0.0016,-0.0003,100.0,518.67,643.78,1596.01,1424.11,14.62,...,0.03,394,2388,100.0,38.44,22.9631,1,2,100_1_2,4
45347,100,149,0.0034,-0.0003,100.0,518.67,643.29,1596.38,1429.14,14.62,...,0.03,395,2388,100.0,38.50,22.9746,1,2,100_1_2,3
45348,100,150,-0.0016,0.0004,100.0,518.67,643.84,1604.53,1431.41,14.62,...,0.03,396,2388,100.0,38.39,23.0682,1,2,100_1_2,2
45349,100,151,-0.0023,0.0004,100.0,518.67,643.94,1597.56,1426.57,14.62,...,0.03,395,2388,100.0,38.31,23.0753,1,2,100_1_2,1


In [41]:
# Logique illustrative — adapte les noms de colonnes aux tiens
DF_U2_Train["RUL"] = (
    DF_U2_Train.groupby("machine_uid")["cycle"].transform("max")
    - DF_U2_Train["cycle"]
)

# Plafonnement à 125 (au-delà, la dégradation n'est pas visible dans les capteurs)
DF_U2_Train["RUL"] = DF_U2_Train["RUL"].clip(upper=125)

## Analyse

In [42]:
# Répartitons des features

Features = ['cycle',
            'Altitude / Mach',
            'Throttle Resolver Angle (TRA)',
            'Altitude pressurisée',
            'Température entrée fan — T2',
            'Température sortie LPC — T24',
            'Température sortie HPC — T30',
            'Température sortie LPT — T50',
            'Pression entrée fan — P2',
            'Pression bypass — P15',
            'Pression sortie HPC — P30',
            'Vitesse fan — Nf',
            'Vitesse cœur — Nc',
            'Rapport pression moteur — EPR',
            'Pression statique HPC — Ps30',
            'Ratio carburant/pression — phi',
            'Vitesse corrigée fan — NRf',
            'Vitesse corrigée cœur — NRc',
            'Bypass Ratio — BPR',
            'Richesse carburant — farB',
            'Soutirage HPC — htBleed',
            'Consigne vitesse fan — Nf_dmd',
            'Consigne vitesse corrigée — PCNfR',
            'Débit refroid. HPT — W31',
            'Débit refroid. LPT — W32',
            'usine','ligne_production',
            'machine_uid',
            'RUL']

Maintenant il faut determiné quelle features sera utiles a gardé. Pour cela on utilise une matrix de corrélation

In [43]:
DF_U1_Train.corr()['RUL']

machine_id                          -0.016096
cycle                               -0.576697
Altitude / Mach                     -0.003158
Throttle Resolver Angle (TRA)       -0.006448
Altitude pressurisée                      NaN
Température entrée fan — T2               NaN
Température sortie LPC — T24        -0.678634
Température sortie HPC — T30        -0.677505
Température sortie LPT — T50        -0.758252
Pression entrée fan — P2                  NaN
Pression bypass — P15               -0.101796
Pression sortie HPC — P30           -0.106384
Vitesse fan — Nf                    -0.592996
Vitesse cœur — Nc                   -0.555331
Rapport pression moteur — EPR       -0.332502
Pression statique HPC — Ps30        -0.785707
Ratio carburant/pression — phi      -0.131264
Vitesse corrigée fan — NRf          -0.594199
Vitesse corrigée cœur — NRc         -0.464015
Bypass Ratio — BPR                  -0.253413
Richesse carburant — farB                 NaN
Soutirage HPC — htBleed           

In [44]:
DF_U1_Train.describe()

,machine_id,cycle,Altitude / Mach,Throttle Resolver Angle (TRA),Altitude pressurisée,Température entrée fan — T2,Température sortie LPC — T24,Température sortie HPC — T30,Température sortie LPT — T50,Pression entrée fan — P2,...,Bypass Ratio — BPR,Richesse carburant — farB,Soutirage HPC — htBleed,Consigne vitesse fan — Nf_dmd,Consigne vitesse corrigée — PCNfR,Débit refroid. HPT — W31,Débit refroid. LPT — W32,usine,ligne_production,RUL
count,45351.000000,45351.000000,45351.000000,45351.000000,45351.0,4.535100e+04,45351.000000,45351.000000,45351.000000,4.535100e+04,...,45351.000000,4.535100e+04,45351.000000,45351.0,45351.0,45351.000000,45351.000000,45351.0,45351.000000,45351.000000
mean,49.939626,125.307049,-0.000017,0.000004,100.0,5.186700e+02,642.559339,1589.190970,1406.501317,1.462000e+01,...,8.417088,3.000000e-02,392.859562,2388.0,100.0,38.910178,23.346022,1.0,1.545082,90.270887
std,29.328476,87.813757,0.002191,0.000294,0.0,1.136881e-13,0.524596,6.622906,9.687784,3.552753e-15,...,0.056212,6.938970e-18,1.698605,0.0,0.0,0.236600,0.141834,0.0,0.497969,41.226148
min,1.000000,1.000000,-0.008700,-0.000600,100.0,5.186700e+02,640.840000,1564.300000,1377.060000,1.462000e+01,...,8.156300,3.000000e-02,388.000000,2388.0,100.0,38.140000,22.872600,1.0,1.000000,0.000000
25%,24.000000,57.000000,-0.001500,-0.000200,100.0,5.186700e+02,642.180000,1584.570000,1399.250000,1.462000e+01,...,8.386200,3.000000e-02,392.000000,2388.0,100.0,38.760000,23.254500,1.0,1.000000,56.000000
50%,50.000000,114.000000,0.000000,0.000000,100.0,5.186700e+02,642.520000,1588.800000,1405.510000,1.462000e+01,...,8.421300,3.000000e-02,393.000000,2388.0,100.0,38.900000,23.342400,1.0,2.000000,113.000000
75%,76.000000,174.000000,0.001500,0.000300,100.0,5.186700e+02,642.900000,1593.440000,1412.680000,1.462000e+01,...,8.453500,3.000000e-02,394.000000,2388.0,100.0,39.050000,23.430100,1.0,2.000000,125.000000
max,100.000000,525.000000,0.008700,0.000700,100.0,5.186700e+02,645.110000,1616.910000,1441.490000,1.462000e+01,...,8.584800,3.000000e-02,400.000000,2388.0,100.0,39.850000,23.950500,1.0,2.000000,125.000000


In [45]:
DF_U1_Train.dtypes

machine_id                             int64
cycle                                  int64
Altitude / Mach                      float64
Throttle Resolver Angle (TRA)        float64
Altitude pressurisée                 float64
Température entrée fan — T2          float64
Température sortie LPC — T24         float64
Température sortie HPC — T30         float64
Température sortie LPT — T50         float64
Pression entrée fan — P2             float64
Pression bypass — P15                float64
Pression sortie HPC — P30            float64
Vitesse fan — Nf                     float64
Vitesse cœur — Nc                    float64
Rapport pression moteur — EPR        float64
Pression statique HPC — Ps30         float64
Ratio carburant/pression — phi       float64
Vitesse corrigée fan — NRf           float64
Vitesse corrigée cœur — NRc          float64
Bypass Ratio — BPR                   float64
Richesse carburant — farB            float64
Soutirage HPC — htBleed                int64
Consigne v

In [46]:
DF_U1_Train.std(numeric_only=True)

machine_id                           2.932848e+01
cycle                                8.781376e+01
Altitude / Mach                      2.190701e-03
Throttle Resolver Angle (TRA)        2.935970e-04
Altitude pressurisée                 0.000000e+00
Température entrée fan — T2          1.136881e-13
Température sortie LPC — T24         5.245963e-01
Température sortie HPC — T30         6.622906e+00
Température sortie LPT — T50         9.687784e+00
Pression entrée fan — P2             3.552753e-15
Pression bypass — P15                1.510314e-02
Pression sortie HPC — P30            2.752963e+00
Vitesse fan — Nf                     1.269044e-01
Vitesse cœur — Nc                    2.097031e+01
Rapport pression moteur — EPR        2.644938e-03
Pression statique HPC — Ps30         2.922927e-01
Ratio carburant/pression — phi       2.586206e+00
Vitesse corrigée fan — NRf           1.270033e-01
Vitesse corrigée cœur — NRc          1.772176e+01
Bypass Ratio — BPR                   5.621184e-02


on peut noté ici 4 colonne qui reste a 0 : 
- usine 
- Consigne vitesse fan — Nf_dmd 
- Consigne vitesse corrigée — PCNfR
- Altitude pressurisée

Grâce au corrélation de RUL on peut éliminer : 
les "Nan"
- Température entrée fan — T2
- Pression entrée fan — P2
- Richesse carburant — farB
et les faible corrélation
- ligne_production
- machine_uid
- Throttle Resolver Angle (TRA)
- Altitude / Mach

In [47]:
# Features utiles : 

Features = ['cycle',
            'Température sortie LPC — T24',
            'Température sortie HPC — T30',
            'Température sortie LPT — T50',
            'Pression bypass — P15',
            'Pression sortie HPC — P30',
            'Vitesse fan — Nf',
            'Vitesse cœur — Nc',
            'Rapport pression moteur — EPR',
            'Pression statique HPC — Ps30',
            'Ratio carburant/pression — phi',
            'Vitesse corrigée fan — NRf',
            'Vitesse corrigée cœur — NRc',
            'Bypass Ratio — BPR',
            'Soutirage HPC — htBleed',
            'Débit refroid. HPT — W31',
            'Débit refroid. LPT — W32']

In [48]:
DF_U1_Train.to_csv('Data/csv/train_usine1.csv', index=False)
DF_U1_Test.to_csv('Data/csv/test_usine1.csv',   index=False)
DF_U1_Rul.to_csv('Data/csv/rul_usine1.csv',     index=False)

DF_U2_Train.to_csv('Data/csv/train_usine2.csv', index=False)
DF_U2_Test.to_csv('Data/csv/test_usine2.csv',   index=False)
DF_U2_Rul.to_csv('Data/csv/rul_usine2.csv',     index=False)